### Back-EMF (Electromotive Force)

The back-EMF is the voltage induced by the rotating permanent magnet flux.

**Peak phase back-EMF:**

$$E_p = \omega_e \cdot \psi_m$$

where:
- $\omega_e$ = electrical angular velocity [rad/s]
- $\psi_m$ = permanent magnet flux linkage [Wb]

**Relationship to mechanical speed:**

$$\omega_e = p \cdot \omega_m = p \cdot \frac{2\pi N}{60}$$

where $p$ is the number of pole pairs and $N$ is mechanical speed in rpm.

**Physical significance:** Back-EMF represents the voltage that must be overcome to drive current through the motor at a given speed. Higher flux and higher speed → higher back-EMF → reduced available voltage for current production (important at high speeds).

## Executive Summary

**Purpose:** Control and simulate AC permanent magnet motors

**What it does:** Analyze PMSM (Permanent Magnet Synchronous Motor) in dq-frame.

### Why It Matters
A control engineer designs field-oriented control (FOC) algorithms and needs steady-state current/torque relationships.

### Quick Start
**Inputs:** Motor parameters (inductances, flux linkage, resistance), operating speed, applied voltages

**Outputs:** dq-axis currents, torque, back-EMF, motor profile (constant-torque and flux-weakening regions)

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** Uses winding concepts from 01_winding.ipynb,

**Downstream (used by):** control system design

In [12]:
#| hide
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

## PMSM (Permanent Magnet Synchronous Machine)

A dq-frame model for AC synchronous machines with permanent magnet excitation.

### dq-Frame Theory

The dq-frame (synchronously rotating reference frame) is the foundation of AC motor control.

**Key concepts:**

- **d-axis (Direct)**: Aligned with the permanent magnet flux direction
- **q-axis (Quadrature)**: 90° ahead of the d-axis in the direction of rotation
- **Benefits**: Transforms AC quantities into DC-like quantities at steady-state, enabling intuitive control

**Voltage equations in dq-frame:**

$$V_d = R_s i_d + \frac{d\psi_d}{dt} - \omega_e \psi_q$$

$$V_q = R_s i_q + \frac{d\psi_q}{dt} + \omega_e \psi_d$$

**Flux linkages:**

$$\psi_d = L_d i_d + \psi_m$$

$$\psi_q = L_q i_q$$

In steady-state (constant speed), the derivatives vanish, yielding the simplified equations used here.

### Inductance and Saliency

**Non-salient (SPM) vs. Salient (IPM) machines:**

- **Surface-Permanent Magnet (SPM)**: Magnets mounted on rotor surface → $L_d \approx L_q$
  - Torque produced primarily by interaction between PM flux and q-axis current
  - Limited flux-weakening capability

- **Interior-Permanent Magnet (IPM)**: Magnets buried in rotor poles → $L_d > L_q$
  - Torque has both PM (excitation) and reluctance (saliency) components
  - Better flux-weakening capability due to saliency

**Torque breakdown:**

$$T_e = \underbrace{\frac{3}{2}p\psi_m i_q}_{\text{Excitation torque}} + \underbrace{\frac{3}{2}p(L_d - L_q)i_d i_q}_{\text{Reluctance torque}}$$

For maximum torque at low speed: set $i_d = 0$ (all current goes to $i_q$)

At high speed (flux-weakening): apply negative $i_d$ to reduce back-EMF and maintain voltage margin

### Flux-Weakening Control

At high speeds, the back-EMF becomes large, limiting the available voltage for current production.

**Voltage constraint:**

$$|\mathbf{V}| = \sqrt{(R_s i_d - \omega_e L_q i_q)^2 + (R_s i_q + \omega_e(L_d i_d + \psi_m))^2} \leq V_b$$

**Flux-weakening strategy:**

Apply negative $i_d$ to weaken the total flux ($\psi_d = L_d i_d + \psi_m$), reducing back-EMF and extending the speed range.

**Operating regions:**

1. **Constant-Torque Region** (low speed, $i_d = 0$): Voltage allows rated current → constant torque output
2. **Flux-Weakening Region** (high speed, $i_d < 0$): Reduced voltage → power limited to rated value

In [13]:
#| export
class PMSMParams:
    """Container for PMSM parameters."""
    def __init__(self, p: int, Ld: float, Lq: float, psi_m: float, Rs: float = 0.0, J: float = 0.0):
        self.p = p
        self.Ld = Ld
        self.Lq = Lq
        self.psi_m = psi_m
        self.Rs = Rs
        self.J = J

    @property
    def is_spm(self) -> bool:
        """True if surface-mount (Ld ≈ Lq)."""
        return np.isclose(self.Ld, self.Lq, rtol=0.05)

### Back-EMF

The back-EMF is proportional to speed and PM flux: $E = \omega_e \cdot \psi_m$

In [14]:
#| export
def back_emf(omega_e: float, psi_m: float) -> float:
    """Peak phase back-EMF."""
    return omega_e * psi_m

In [15]:
emf = back_emf(1000, 0.15)
assert emf == 150.0
print(f"✓ Back-EMF test passed")

✓ Back-EMF test passed


### Electromagnetic Torque

The instantaneous torque has two components:

$$T_e = \frac{3}{2} p [\psi_m i_q + (L_d - L_q) i_d i_q]$$

**Component breakdown:**

1. **Excitation (PM) torque:** $T_{exc} = \frac{3}{2} p \psi_m i_q$
   - Produced by interaction of PM flux with q-axis current
   - Present in all PMSM topologies
   - Maximized at $i_d = 0$ for a given current magnitude

2. **Reluctance (Saliency) torque:** $T_{rel} = \frac{3}{2} p (L_d - L_q) i_d i_q$
   - Produced by saliency (magnetic asymmetry)
   - Only non-zero in IPM machines where $L_d > L_q$
   - Requires specific $i_d$ and $i_q$ signs to add constructively

**Control implications:**
- **Low speed (constant-torque region):** Use $i_d = 0$, vary $i_q$ for torque control
- **High speed (flux-weakening region):** Use $i_d < 0$ to reduce back-EMF and maintain power output

In [16]:
#| export
def torque(params: PMSMParams, id: float, iq: float) -> float:
    """Electromagnetic torque."""
    T_excitation = params.psi_m * iq
    T_reluctance = (params.Ld - params.Lq) * id * iq
    return 1.5 * params.p * (T_excitation + T_reluctance)

In [17]:
params = PMSMParams(p=5, Ld=78e-6, Lq=78e-6, psi_m=0.15)
T = torque(params, 0, 100)
assert T > 0
print(f"✓ PMSM torque test passed")

✓ PMSM torque test passed


### Steady-State dq-Frame Currents

Solve the 2×2 linear system for steady-state dq currents from applied voltages.

**Steady-state voltage equations (neglecting transients):**

$$V_d = R_s i_d - \omega_e L_q i_q$$

$$V_q = R_s i_q + \omega_e (L_d i_d + \psi_m)$$

**Physical interpretation:**
- **d-axis:** Resistive voltage drop plus cross-coupling from q-axis current
- **q-axis:** Resistive drop plus back-EMF and cross-coupling from d-axis current

**Matrix form:**

$$\begin{bmatrix} V_d \ V_q - \omega_e\psi_m \end{bmatrix} = \begin{bmatrix} R_s & -\omega_e L_q \ \omega_e L_d & R_s \end{bmatrix} \begin{bmatrix} i_d \ i_q \end{bmatrix}$$

Solving this system gives the steady-state currents needed to produce a given voltage vector.

In [18]:
#| export
def dq_currents(params: PMSMParams, omega_e: float, Vd: float, Vq: float) -> tuple[float, float]:
    """Steady-state dq currents."""
    A = np.array([[params.Rs, -omega_e * params.Lq],
                  [omega_e * params.Ld, params.Rs]])
    b = np.array([Vd, Vq - omega_e * params.psi_m])
    id_, iq_ = np.linalg.solve(A, b)
    return float(id_), float(iq_)

In [19]:
params = PMSMParams(p=5, Ld=78e-6, Lq=78e-6, psi_m=0.15, Rs=0.1)
id, iq = dq_currents(params, 500, 2.0, 10.0)
assert not np.isnan(id) and not np.isnan(iq)
print(f"✓ DQ-frame currents test passed")

✓ DQ-frame currents test passed


### SPM Motor Profile

Surface-mount PMSM with constant-torque and flux-weakening regions.

In [20]:
#| export
class SPM:
    """Surface-Mount PMSM motor profile."""
    def __init__(self, phi_m: float, ld: float, pp: int, Vb: float, Ib: float):
        self.phi_m = phi_m
        self.sal = 0
        self.ld = ld
        self.Vb = Vb
        self.Ib = Ib
        self.Pb = 1.5 * Vb * Ib
        self.pp = pp
        self.speed = []
        self.torque = []
        self.voltage = []
        self.gamma = []
        self.power = []
        self.valid = 0

    def validate(self) -> bool:
        """Check parameter validity."""
        self.valid = 1 if self.sal == 0 else 0
        return self.valid == 1

    def motor_profile(self) -> None:
        """Compute torque-speed profile."""
        self.speed = []
        self.torque = []
        self.voltage = []
        self.gamma = []
        self.power = []
        gamma_deg = 0
        gamma = 0.0
        omega = 0.0
        v = 0.0
        v_lim = self.Vb / 1.732
        while v < v_lim:
            v = omega * self.pp * np.sqrt(
                (self.phi_m - self.Ib * np.sin(gamma) * self.ld) ** 2
                + (self.Ib * self.ld * np.cos(gamma)) ** 2
            )
            t = 1.5 * self.pp * self.phi_m * self.Ib * np.cos(gamma)
            p = t * omega
            omega += 0.1
            self.speed.append(omega * 30 / np.pi)
            self.voltage.append(v)
            self.gamma.append(gamma_deg)
            self.torque.append(t)
            self.power.append(p)
        for gamma_deg in range(1, 85):
            gamma = gamma_deg * np.pi / 180
            omega = (v_lim) / (self.pp * np.sqrt(
                (self.phi_m - self.Ib * np.sin(gamma) * self.ld) ** 2
                + (self.Ib * self.ld * np.cos(gamma)) ** 2
            ))
            t = 1.5 * self.pp * self.Ib * self.phi_m * np.cos(gamma)
            p = t * omega
            self.speed.append(omega * 30 / np.pi)
            self.voltage.append(v)
            self.gamma.append(gamma_deg)
            self.torque.append(t)
            self.power.append(p)

In [21]:
spm = SPM(phi_m=0.087, ld=78e-6, pp=5, Vb=12, Ib=50)
spm.validate()
spm.motor_profile()
assert len(spm.torque) == len(spm.speed)
print(f"✓ SPM motor profile test passed")

✓ SPM motor profile test passed


## Tests

PMSM dq-model validated against analytical expressions.

In [ ]:
#| hide
import numpy as np, math

# ── back_emf ──────────────────────────────────────────────────────────
spm = PMSMParams(p=3, Ld=5e-3, Lq=5e-3, psi_m=0.15, Rs=0.5)
omega_e = 2*np.pi*50 * spm.p
E = back_emf(omega_e, spm.psi_m)
assert E > 0
assert np.isclose(E, omega_e * spm.psi_m)
print("✓ back_emf")

# ── SPM: no reluctance torque ─────────────────────────────────────────
Te_1 = torque(spm, id=0.0, iq=10.0)
Te_2 = torque(spm, id=-5.0, iq=10.0)
assert np.isclose(Te_1, Te_2, atol=1e-10), "SPM: id should not affect torque"

# ── IPM: reluctance torque boost ─────────────────────────────────────
ipm = PMSMParams(p=4, Ld=3e-3, Lq=8e-3, psi_m=0.12, Rs=0.3)
Te_id0 = torque(ipm, id=0.0, iq=10.0)
Te_mtpa = torque(ipm, id=-5.0, iq=10.0)
assert Te_mtpa > Te_id0, "IPM: negative id should add reluctance torque"

# ── torque sign ───────────────────────────────────────────────────────
assert torque(spm, id=0.0, iq=5.0) > 0
assert torque(spm, id=0.0, iq=-5.0) < 0
print("✓ torque (SPM, IPM, sign)")

# ── dq_currents: steady-state voltage equation verification ───────────
omega_e = 2*np.pi*100.0
Vd, Vq = 0.0, 100.0
id_, iq_ = dq_currents(spm, omega_e, Vd, Vq)
Vd_chk = spm.Rs*id_ - omega_e*spm.Lq*iq_
Vq_chk = spm.Rs*iq_ + omega_e*(spm.Ld*id_ + spm.psi_m)
assert np.isclose(Vd_chk, Vd, atol=1e-6), f"Vd mismatch: {Vd_chk}"
assert np.isclose(Vq_chk, Vq, atol=1e-6), f"Vq mismatch: {Vq_chk}"
print("✓ dq_currents")
print("\n✓ All PMSM tests passed")
